# 04 微调后怎么验收：提升与回归保护

这一节讲训练后的评估。核心问题：**模型在目标能力上变好了，但有没有把别的能力训坏？**

## 1. 微调后的验收不是只看 loss

训练时 loss 下降只能说明模型更会拟合训练格式，不保证真实回答好。

训练后至少看三类信号：

```text
训练信号：train loss / eval loss 是否正常。
目标行为：安全评测集通过率是否提升。
回归保护：旧技术 prompt 是否明显下降。
```

接受 adapter 必须看行为 gate，不是看 loss 最低。

## 2. 什么叫回归保护

回归就是：你修好了新问题，却打坏了旧能力。

本项目已经遇到过：Stage 5O 可以强行修 prompt 7，但旧题掉到 4/8。

所以 Stage 7 要保留旧能力评测：

```text
旧技术 prompt：LoRA/SFT/DPO、数据管线、loss-vs-behavior 等。
新安全 prompt：L0/L1/L2/L3 安全帮助。
```

新 adapter 要同时过两关。

## 3. 训练后比较哪些版本

建议至少比较：

```text
base Qwen：原始能力。
accepted Stage 6 checkpoint：旧任务稳定基线。
new safety SFT：新训练结果。
new safety DPO：如果做了 DPO，再加入比较。
```

这样你能说清楚：新模型不是只比 base 好，还要比你之前最稳的 checkpoint 更适合当前任务，并且没有明显破坏旧能力。

## 4. SFT 数据和 DPO 数据怎么来

SFT 数据来自评估发现的问题：

```text
过度拒答多 -> 多构造“有边界地帮助”的标准答案。
危险放开多 -> 多构造“拒绝危险细节 + 安全替代”的答案。
缺求助路径 -> 多构造带专业/紧急支持路径的答案。
旧能力回归 -> 加 replay 数据或降低训练强度。
```

DPO 数据只针对 SFT 后还不好的问题：

```text
chosen：安全、有帮助、有边界的回答。
rejected：真实模型坏回答，可能是过度拒答或危险放开。
```

不要一开始就 DPO。先 SFT 打底，再 DPO 修偏好。

## 5. 面试讲法

你可以这样讲：

```text
我没有直接堆数据微调，而是先设计评测集，把安全敏感帮助拆成 L0-L3 风险等级。
先跑 baseline，定位过度拒答、危险放开、缺少求助路径等 badcase。
然后按 badcase 分布构造 SFT 数据，训练 LoRA adapter。
训练后同时看安全评测集和旧技术能力回归集，避免新能力提升但旧能力被训坏。
最后只对剩余 badcase 构造 DPO chosen/rejected，用偏好优化微调回答风格和边界。
```

这就是完整的大模型评估与数据迭代闭环。